# Lab 04 — Baseline Evaluation

**Goal:** measure how well the untouched `Qwen/Qwen3-0.6B` model actually plays Wordle before we train it on Wordle.

This lab is where anecdotes become a benchmark.

By the end, you should have:

- a fixed evaluation answer set;
- deterministic model generation;
- a strict guess parser;
- a game loop that never reveals the hidden answer;
- per-game traces;
- aggregate metrics;
- a failure taxonomy;
- a baseline result we can compare against every later checkpoint.

We will use **non-thinking mode** and **greedy decoding** for the primary baseline. Qwen exposes `enable_thinking=False`, and Hugging Face defines `do_sample=False` with one beam as greedy decoding. That gives us a reproducible baseline rather than a lottery.

## 4.1 Important setup: reload the untouched checkpoint

Lab 02 modified Qwen's weights in memory.

Do **not** benchmark that model.

A baseline must begin from the original pretrained checkpoint. Restarting the Jupyter kernel is the safest option. Then run this notebook from the top.

If you do not restart, explicitly reload `Qwen/Qwen3-0.6B` from disk/Hugging Face before continuing.

In [1]:
import re
import time
from dataclasses import dataclass, asdict
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from tiny_wordle.game import Turn, score_string

# MODEL_ID = "../checkpoints/qwen3-0.6b-wordle-full-sft" # Lab 07
# MODEL_ID = "../checkpoints/qwen3-0.6b-wordle-onpolicy-sft-v2" # Lab 08 on polocy
# MODEL_ID = "../checkpoints/qwen3-0.6b-wordle-gameplay-policy-sft" # lab 08 gameplay
MODEL_ID = (
    "../checkpoints/"
    "qwen3-0.6b-wordle-lora-merged"
) # Lab 08 Lora

if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("device:", device)

device: mps


In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float32,
).to(device)

model.eval()

print(type(model).__name__)
print("parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("device:", next(model.parameters()).device)
print("dtype:", next(model.parameters()).dtype)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Qwen3ForCausalLM
parameters: 596,049,920
device: mps:0
dtype: torch.float32


## 4.2 Freeze the first evaluation answer set

We need a fixed answer set before changing prompts or training.

For this first benchmark, we will use a small, explicit set so that the notebook is easy to inspect and fast enough to iterate on. Later we will replace this with a larger frozen benchmark file.

These answers must **never become training examples** in later labs.

A 20-word baseline is not enough for a final scientific claim. It *is* enough to debug the evaluation harness.

In [3]:
EVAL_ANSWERS = [
    "PLANT",
    "SHORE",
    "MIGHT",
    "BRICK",
    "GHOST",
    "KNIFE",
    "DOUBT",
    "FLING",
    "ROUND",
    "CHAMP",
    "WASTE",
    "BLIND",
    "POINT",
    "SLATE",
    "CRANE",
    "APPLE",
    "SHEEP",
    "BANAL",
    "ALLEY",
    "AUDIO",
]

assert len(EVAL_ANSWERS) == len(set(EVAL_ANSWERS))
assert all(len(word) == 5 and word.isalpha() and word.isupper() for word in EVAL_ANSWERS)

print("evaluation answers:", len(EVAL_ANSWERS))

evaluation answers: 20


### Leakage rule

Write this down:

> **These 20 answers are evaluation-only from this point forward.**

Later, when we generate synthetic training data, we will exclude them.

This is the first place where benchmark discipline matters more than code cleverness.

## 4.3 Decide exactly what the model is told

The model never sees the hidden answer.

On turn 1 it gets the game rules and no history.

On later turns it gets only previous guesses and their exact `B/Y/G` feedback.

We deliberately use the spaced-letter representation because Lab 01 showed that it aligns more cleanly with Qwen's tokenizer. This is now part of the baseline configuration, so we must keep it fixed when comparing later checkpoints unless representation itself is the experiment.

In [4]:
SYSTEM_RULES = """Play Wordle.

Return exactly one uppercase five-letter English word.
Do not explain.
Do not use punctuation.

Use all previous guesses and feedback when choosing the next guess.
Never repeat a previous guess.

Example valid response:
CRANE

Feedback meanings:
G = correct letter and position
Y = letter is present but wrong position
B = that letter occurrence is not matched
"""

def format_history_for_model(history: list[Turn]) -> str:
    if not history:
        return "No guesses have been made yet."

    lines = []
    for turn in history:
        spaced_guess = " ".join(turn.guess)
        spaced_feedback = " ".join(turn.feedback)
        lines.append(f"{spaced_guess} -> {spaced_feedback}")

    return "\n".join(lines)

def build_prompt(history: list[Turn]) -> str:
    return SYSTEM_RULES + "\nGame history:\n" + format_history_for_model(history)

print(build_prompt([]))

Play Wordle.

Return exactly one uppercase five-letter English word.
Do not explain.
Do not use punctuation.

Use all previous guesses and feedback when choosing the next guess.
Never repeat a previous guess.

Example valid response:
CRANE

Feedback meanings:
G = correct letter and position
Y = letter is present but wrong position
B = that letter occurrence is not matched

Game history:
No guesses have been made yet.


## 4.4 Strictly parse model output

Lab 01 showed that Qwen may answer with prose, markdown, or the wrong number of letters.

We need to decide what counts as a usable action.

For the primary benchmark, the parser will be deliberately strict:

- strip whitespace;
- accept only a single alphabetic token;
- require exactly five letters;
- uppercase it.

This means `CRANE` is valid.

`Guess: CRANE`, `**CRANE**`, `cute`, and `answer` are invalid.

Why strict? Because instruction following is part of the capability we are measuring. A parser that heroically excavates a word from bad prose would hide a real model failure.

In [5]:
WORD_RE = re.compile(r"^[A-Za-z]{5}$")

def parse_guess(raw_text: str) -> str | None:
    text = raw_text.strip()

    if not WORD_RE.fullmatch(text):
        return None

    return text.upper()

tests = {
    "CRANE": "CRANE",
    " crane ": "CRANE",
    "Guess: CRANE": None,
    "**CRANE**": None,
    "cute": None,
    "answer": None,
    "BAY": None,
}

for raw, expected in tests.items():
    actual = parse_guess(raw)
    assert actual == expected, (raw, expected, actual)

print("Parser tests passed.")

Parser tests passed.


## 4.5 Generate one guess deterministically

We use:

```python
enable_thinking=False
do_sample=False
```

That keeps the main baseline reproducible.

Qwen's own model card provides a hard switch for disabling thinking. Hugging Face generation docs define this `do_sample=False` configuration as greedy decoding.

Later we can separately test whether thinking or sampling helps. Do not mix that into the baseline yet.

In [6]:
def generate_raw_guess(history: list[Turn], max_new_tokens: int = 16) -> str:
    user_text = build_prompt(history)

    chat_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_text}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    batch = tokenizer(chat_text, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **batch,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    new_tokens = output[0, batch["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

raw = generate_raw_guess([])
print(repr(raw))
print("parsed:", parse_guess(raw))

'BANAN'
parsed: BANAN


Stop here and inspect the first output.

If the model fails the strict parser on turn 1, that is not a bug in the benchmark. It is a baseline capability failure.

We still need a policy for how the game proceeds after an invalid response.

## 4.6 Invalid responses consume a turn

A real player who fails to submit a legal Wordle guess does not get useful feedback.

For our benchmark:

- every model call consumes one of the six turns;
- invalid output gets no game feedback;
- the same history is shown again on the next turn.

This makes format failures costly without inventing fake feedback.

We also distinguish:
- `invalid_format`: not exactly one five-letter alphabetic word;
- `repeat_guess`: valid format but already guessed before.

A repeated guess is still a legal-looking action, but strategically useless. We will score it against the hidden answer normally.

In [7]:
@dataclass
class GameResult:
    answer: str
    solved: bool
    turns_used: int
    valid_guesses: int
    invalid_format: int
    repeat_guesses: int
    final_guess: str | None
    elapsed_seconds: float
    trace: list[dict]


def play_model_game(answer: str, max_turns: int = 6, verbose: bool = False) -> GameResult:
    answer = answer.upper()
    history: list[Turn] = []
    seen: set[str] = set()
    trace = []

    invalid_format = 0
    repeat_guesses = 0
    start = time.perf_counter()

    for turn_number in range(1, max_turns + 1):
        raw = generate_raw_guess(history)
        guess = parse_guess(raw)

        record = {
            "turn": turn_number,
            "raw_output": raw,
            "guess": guess,
            "feedback": None,
            "status": None,
        }

        if guess is None:
            invalid_format += 1
            record["status"] = "invalid_format"
            trace.append(record)

            if verbose:
                print(f"{turn_number}: INVALID {raw!r}")

            continue

        if guess in seen:
            repeat_guesses += 1
            record["status"] = "repeat_guess"
        else:
            record["status"] = "valid_guess"

        seen.add(guess)

        feedback = score_string(answer, guess)
        record["feedback"] = feedback
        trace.append(record)
        history.append(Turn(guess=guess, feedback=feedback))

        if verbose:
            print(f"{turn_number}: {guess} -> {feedback}")

        if feedback == "GGGGG":
            elapsed = time.perf_counter() - start
            return GameResult(
                answer=answer,
                solved=True,
                turns_used=turn_number,
                valid_guesses=len(history),
                invalid_format=invalid_format,
                repeat_guesses=repeat_guesses,
                final_guess=guess,
                elapsed_seconds=elapsed,
                trace=trace,
            )

    elapsed = time.perf_counter() - start

    return GameResult(
        answer=answer,
        solved=False,
        turns_used=max_turns,
        valid_guesses=len(history),
        invalid_format=invalid_format,
        repeat_guesses=repeat_guesses,
        final_guess=history[-1].guess if history else None,
        elapsed_seconds=elapsed,
        trace=trace,
    )

## 4.7 Play one game before running the benchmark

Choose one evaluation answer and inspect the full trace.

Do not run all 20 games until one game behaves sensibly. Debugging a batch of broken evaluations is just industrializing confusion.

In [8]:
one_game = play_model_game(EVAL_ANSWERS[0], verbose=True)
one_game

1: BANAN -> BYYBB
2: BANAN -> BYYBB
3: BANAN -> BYYBB
4: BANAN -> BYYBB
5: BANAN -> BYYBB
6: BANAN -> BYYBB


GameResult(answer='PLANT', solved=False, turns_used=6, valid_guesses=6, invalid_format=0, repeat_guesses=5, final_guess='BANAN', elapsed_seconds=0.5173670410295017, trace=[{'turn': 1, 'raw_output': 'BANAN', 'guess': 'BANAN', 'feedback': 'BYYBB', 'status': 'valid_guess'}, {'turn': 2, 'raw_output': 'BANAN', 'guess': 'BANAN', 'feedback': 'BYYBB', 'status': 'repeat_guess'}, {'turn': 3, 'raw_output': 'BANAN', 'guess': 'BANAN', 'feedback': 'BYYBB', 'status': 'repeat_guess'}, {'turn': 4, 'raw_output': 'BANAN', 'guess': 'BANAN', 'feedback': 'BYYBB', 'status': 'repeat_guess'}, {'turn': 5, 'raw_output': 'BANAN', 'guess': 'BANAN', 'feedback': 'BYYBB', 'status': 'repeat_guess'}, {'turn': 6, 'raw_output': 'BANAN', 'guess': 'BANAN', 'feedback': 'BYYBB', 'status': 'repeat_guess'}])

Inspect:

- Did the hidden answer ever appear in the prompt? It must not.
- Did each valid guess receive correct feedback?
- Did an invalid response consume a turn but leave history unchanged?
- Did a repeated valid guess get scored normally?
- Did the game stop immediately on `GGGGG`?

If any answer is no, stop and fix the harness.

## 4.8 Run the baseline

Now evaluate the fixed answer set.

This can take a little while because each game can require up to six autoregressive generations on MPS.

We save traces, not just aggregate metrics. When a number looks strange, we need to be able to inspect why.

In [9]:
results = []

for i, answer in enumerate(EVAL_ANSWERS, 1):
    result = play_model_game(answer)
    results.append(result)

    status = "SOLVED" if result.solved else "FAILED"
    print(
        f"{i:2d}/{len(EVAL_ANSWERS)} "
        f"{answer} {status:6s} "
        f"turns={result.turns_used} "
        f"valid={result.valid_guesses} "
        f"invalid={result.invalid_format} "
        f"repeats={result.repeat_guesses}"
    )

 1/20 PLANT FAILED turns=6 valid=6 invalid=0 repeats=5
 2/20 SHORE FAILED turns=6 valid=6 invalid=0 repeats=5
 3/20 MIGHT FAILED turns=6 valid=6 invalid=0 repeats=5
 4/20 BRICK FAILED turns=6 valid=6 invalid=0 repeats=5
 5/20 GHOST FAILED turns=6 valid=6 invalid=0 repeats=5
 6/20 KNIFE FAILED turns=6 valid=6 invalid=0 repeats=5
 7/20 DOUBT FAILED turns=6 valid=6 invalid=0 repeats=5
 8/20 FLING FAILED turns=6 valid=6 invalid=0 repeats=5
 9/20 ROUND FAILED turns=6 valid=6 invalid=0 repeats=5
10/20 CHAMP FAILED turns=6 valid=6 invalid=0 repeats=5
11/20 WASTE FAILED turns=6 valid=6 invalid=0 repeats=5
12/20 BLIND FAILED turns=6 valid=6 invalid=0 repeats=5
13/20 POINT FAILED turns=6 valid=6 invalid=0 repeats=5
14/20 SLATE FAILED turns=6 valid=6 invalid=0 repeats=5
15/20 CRANE FAILED turns=6 valid=6 invalid=0 repeats=5
16/20 APPLE FAILED turns=6 valid=6 invalid=0 repeats=5
17/20 SHEEP FAILED turns=6 valid=6 invalid=0 repeats=5
18/20 BANAL FAILED turns=6 valid=6 invalid=0 repeats=5
19/20 ALLE

## 4.9 Aggregate metrics

We care about more than win rate.

A model can fail because it:

- cannot format a legal guess;
- repeats itself;
- produces plausible guesses but ignores feedback;
- follows constraints yet chooses poor strategy.

The baseline should expose those separately.

In [10]:
rows = []

for r in results:
    rows.append({
        "answer": r.answer,
        "solved": r.solved,
        "turns_used": r.turns_used,
        "valid_guesses": r.valid_guesses,
        "invalid_format": r.invalid_format,
        "repeat_guesses": r.repeat_guesses,
        "final_guess": r.final_guess,
        "elapsed_seconds": r.elapsed_seconds,
    })

df = pd.DataFrame(rows)
df

,answer,solved,turns_used,valid_guesses,invalid_format,repeat_guesses,final_guess,elapsed_seconds
0,PLANT,False,6,6,0,5,BANAN,0.496031
1,SHORE,False,6,6,0,5,BANAN,0.491828
2,MIGHT,False,6,6,0,5,BANAN,0.489132
3,BRICK,False,6,6,0,5,BANAN,0.485507
4,GHOST,False,6,6,0,5,BANAN,0.487971
5,KNIFE,False,6,6,0,5,BANAN,0.486867
6,DOUBT,False,6,6,0,5,BANAN,0.489774
7,FLING,False,6,6,0,5,BANAN,0.485756
8,ROUND,False,6,6,0,5,BANAN,0.488585
9,CHAMP,False,6,6,0,5,BANAN,0.486115


In [11]:
games = len(df)
solves = int(df["solved"].sum())
total_model_calls = int(df["turns_used"].sum())
total_valid = int(df["valid_guesses"].sum())
total_invalid = int(df["invalid_format"].sum())
total_repeats = int(df["repeat_guesses"].sum())

solve_rate = solves / games
valid_output_rate = total_valid / total_model_calls if total_model_calls else 0.0
invalid_output_rate = total_invalid / total_model_calls if total_model_calls else 0.0
mean_turns_on_wins = df.loc[df["solved"], "turns_used"].mean()

print(f"games:                {games}")
print(f"solved:               {solves}")
print(f"solve rate:           {solve_rate:.1%}")
print(f"model calls:          {total_model_calls}")
print(f"valid output rate:    {valid_output_rate:.1%}")
print(f"invalid output rate:  {invalid_output_rate:.1%}")
print(f"repeat guesses:       {total_repeats}")
print(f"mean turns on wins:   {mean_turns_on_wins}")
print(f"total eval time:      {df['elapsed_seconds'].sum():.1f}s")

games:                20
solved:               0
solve rate:           0.0%
model calls:          120
valid output rate:    100.0%
invalid output rate:  0.0%
repeat guesses:       100
mean turns on wins:   nan
total eval time:      9.8s


## 4.10 Measure feedback consistency

A valid five-letter guess is not necessarily a **consistent** guess.

Example:

```text
CRANE -> BBGGB
```

tells us certain facts about the answer. A later guess that contradicts those facts demonstrates state-tracking failure.

We can test this exactly.

For each valid guess after turn 1, ask:

> Is this guessed word itself consistent with all prior feedback?

This is not the same as strategic quality. A good Wordle player may intentionally use a probe word that violates hard-mode constraints to gain information. So call this **history consistency**, not legality.

For our eventual specialist, we will decide explicitly whether we want hard-mode-style constraint adherence or unrestricted Wordle strategy.

In [12]:
def history_consistency_stats(result: GameResult) -> tuple[int, int]:
    prior_history: list[Turn] = []
    checked = 0
    consistent = 0

    for step in result.trace:
        guess = step["guess"]

        if guess is None:
            continue

        if prior_history:
            checked += 1

            matches_all_prior_feedback = all(
                score_string(guess, old.guess) == old.feedback
                for old in prior_history
            )

            consistent += int(matches_all_prior_feedback)

        if step["feedback"] is not None:
            prior_history.append(
                Turn(
                    guess=guess,
                    feedback=step["feedback"],
                )
            )

    return consistent, checked


consistent_total = 0
checked_total = 0

for result in results:
    consistent, checked = history_consistency_stats(result)
    consistent_total += consistent
    checked_total += checked

history_consistency_rate = (
    consistent_total / checked_total
    if checked_total
    else float("nan")
)

print("post-first-turn valid guesses checked:", checked_total)
print("history-consistent guesses:", consistent_total)
print(f"history consistency rate: {history_consistency_rate:.1%}")

post-first-turn valid guesses checked: 100
history-consistent guesses: 0
history consistency rate: 0.0%


### Important caveat

Do not overinterpret this metric.

Normal Wordle allows informational guesses that may deliberately ignore known positions. So a low history-consistency rate can mean either:

1. the model forgot the state; or
2. the model intentionally played an exploratory guess.

Given what we observed in Lab 01, explanation #1 is likely common for Qwen3-0.6B—but we should inspect traces instead of assuming.

## 4.11 Inspect failures

Aggregate metrics tell us **that** the model failed. Traces tell us **how**.

Print every failed game.

In [13]:
for result in results:
    if result.solved:
        continue

    print("=" * 70)
    print("ANSWER:", result.answer)

    for step in result.trace:
        print(
            f"turn={step['turn']} "
            f"status={step['status']} "
            f"raw={step['raw_output']!r} "
            f"guess={step['guess']} "
            f"feedback={step['feedback']}"
        )

ANSWER: PLANT
turn=1 status=valid_guess raw='BANAN' guess=BANAN feedback=BYYBB
turn=2 status=repeat_guess raw='BANAN' guess=BANAN feedback=BYYBB
turn=3 status=repeat_guess raw='BANAN' guess=BANAN feedback=BYYBB
turn=4 status=repeat_guess raw='BANAN' guess=BANAN feedback=BYYBB
turn=5 status=repeat_guess raw='BANAN' guess=BANAN feedback=BYYBB
turn=6 status=repeat_guess raw='BANAN' guess=BANAN feedback=BYYBB
ANSWER: SHORE
turn=1 status=valid_guess raw='BANAN' guess=BANAN feedback=BBBBB
turn=2 status=repeat_guess raw='BANAN' guess=BANAN feedback=BBBBB
turn=3 status=repeat_guess raw='BANAN' guess=BANAN feedback=BBBBB
turn=4 status=repeat_guess raw='BANAN' guess=BANAN feedback=BBBBB
turn=5 status=repeat_guess raw='BANAN' guess=BANAN feedback=BBBBB
turn=6 status=repeat_guess raw='BANAN' guess=BANAN feedback=BBBBB
ANSWER: MIGHT
turn=1 status=valid_guess raw='BANAN' guess=BANAN feedback=BBBBB
turn=2 status=repeat_guess raw='BANAN' guess=BANAN feedback=BBBBB
turn=3 status=repeat_guess raw='BANAN

## 4.12 Assign failure categories

For now we will categorize at the game level.

This is intentionally simple. Later we can automate more of the taxonomy.

Suggested categories:

- `FORMAT`: too many invalid outputs to play effectively
- `REPETITION`: wastes turns repeating guesses
- `STATE`: valid guesses visibly contradict prior feedback
- `STRATEGY`: generally tracks state but fails within six
- `SOLVED`: solved

The point is to separate capability layers rather than collapse everything into win/loss.

In [14]:
def categorize_game(result: GameResult) -> str:
    if result.solved:
        return "SOLVED"

    if result.invalid_format >= 2:
        return "FORMAT"

    if result.repeat_guesses >= 2:
        return "REPETITION"

    consistent, checked = history_consistency_stats(result)
    if checked and consistent / checked < 0.5:
        return "STATE"

    return "STRATEGY"


categories = [categorize_game(r) for r in results]
category_counts = pd.Series(categories).value_counts()

category_counts

REPETITION    20
Name: count, dtype: int64

## 4.13 Save the baseline artifacts

A benchmark we cannot reproduce later is barely better than an anecdote.

Save:

- summary table;
- full per-turn traces;
- benchmark configuration.

The files go under `results/`.

In [15]:
import json

results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)

summary_path = results_dir / "baseline_qwen3_0.6b_summary.csv"
trace_path = results_dir / "baseline_qwen3_0.6b_traces.json"
config_path = results_dir / "baseline_qwen3_0.6b_config.json"

df.assign(category=categories).to_csv(summary_path, index=False)

with trace_path.open("w") as f:
    json.dump([asdict(r) for r in results], f, indent=2)

config = {
    "model": MODEL_ID,
    "dtype": "float32",
    "device": str(device),
    "thinking": False,
    "do_sample": False,
    "max_turns": 6,
    "parser": "strict_single_5_letter_alpha_token",
    "representation": "spaced_letters_and_spaced_BYG",
    "eval_answers": EVAL_ANSWERS,
}

with config_path.open("w") as f:
    json.dump(config, f, indent=2)

print(summary_path)
print(trace_path)
print(config_path)

../results/baseline_qwen3_0.6b_summary.csv
../results/baseline_qwen3_0.6b_traces.json
../results/baseline_qwen3_0.6b_config.json


## 4.14 Baseline report

Fill this in from the actual numbers.

Do not round an ugly baseline into something flattering. The purpose of the baseline is to be beaten later.

Record:

```text
Model:
Checkpoint:
Thinking mode:
Decoding:
Representation:
Evaluation games:

Solve rate:
Valid output rate:
Invalid output rate:
History consistency rate:
Repeat guesses:
Mean turns among wins:

Primary failure category:
Most surprising failure:
```

## 4.15 Optional diagnostic: relaxed parser

Do **not** replace the primary benchmark parser.

But one useful diagnostic is to ask:

> How much performance are we losing purely because Qwen wraps an otherwise valid word in prose or markdown?

A relaxed parser can extract the first standalone five-letter word. Run this only as a secondary diagnostic.

If relaxed parsing causes a large jump, formatting is a major bottleneck.

If it barely matters, the deeper problem is state/strategy.

In [16]:
RELAXED_WORD_RE = re.compile(r"\b[A-Za-z]{5}\b")

def parse_guess_relaxed(raw_text: str) -> str | None:
    match = RELAXED_WORD_RE.search(raw_text)
    return match.group(0).upper() if match else None

examples = [
    "CRANE",
    "Guess: CRANE",
    "**CRANE**",
    "One possible guess is SLATE.",
    "cute",
    "answer",
]

for x in examples:
    print(repr(x), "strict=", parse_guess(x), "relaxed=", parse_guess_relaxed(x))

'CRANE' strict= CRANE relaxed= CRANE
'Guess: CRANE' strict= None relaxed= GUESS
'**CRANE**' strict= None relaxed= CRANE
'One possible guess is SLATE.' strict= None relaxed= GUESS
'cute' strict= None relaxed= None
'answer' strict= None relaxed= None


Do not rerun the full benchmark with relaxed parsing unless we explicitly decide to make that an experiment. Otherwise we will start tuning the benchmark while looking at its answers.

That road ends with excellent metrics and no idea what they mean.

## Lab 04 checkpoint

You should now be able to explain:

1. Why we reloaded the untouched checkpoint after Lab 02.
2. Why an evaluation answer set must be frozen before training.
3. Why greedy non-thinking decoding is useful for a reproducible primary baseline.
4. Why strict parsing intentionally counts some recoverable outputs as failures.
5. Why invalid model output consumes a turn without generating feedback.
6. Why win rate alone is insufficient.
7. The difference between format failure, state-tracking failure, and strategic failure.
8. Why history consistency is useful but not identical to good Wordle strategy.
9. Why traces and benchmark configuration must be saved.

### Send me these outputs

First, do **not** run all 20 games immediately.

Run through section **4.7 Play one game** and send me:

- the raw first-turn generation from 4.5;
- the parsed value;
- the full one-game trace from 4.7.

We'll inspect that before launching the complete baseline.

### What comes next

**Lab 05 — Symbolic Expert**

Once the baseline is frozen, we will build a strong non-neural player. That expert will provide:

- a performance ceiling;
- exact candidate sets;
- high-quality state → action demonstrations;
- synthetic supervision for SFT and distillation.